<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/python-primer-notebooks/blob/main/18-visualisation.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/python-primer/) collection._

# Chapter 18 — Visualisation That Isn't Decoration

Companion to [the chapter](https://www.ai.biz/books/python-primer/visualisation/).


In [ ]:
import matplotlib
matplotlib.use('Agg')   # remove this line in Colab
import matplotlib.pyplot as plt
import pandas as pd, numpy as np
rng = np.random.default_rng(0)


In [ ]:
n = 2000
df = pd.DataFrame({
    'age': rng.normal(42, 12, n).clip(18, 90).round(),
    'revenue': rng.lognormal(4, 1, n).round(2),
    'tier': rng.choice(['gold','silver','bronze'], n, p=[.2,.3,.5]),
    'region': rng.choice(['North','South','East','West'], n),
})
dates = pd.date_range('2026-01-01', periods=180, freq='D')
daily = pd.Series(rng.normal(1000, 150, 180).cumsum() / 100, index=dates, name='revenue')
df.head()


## 1. Two APIs. Use the object-oriented one.


In [ ]:
# The stateful API — breaks as soon as there is more than one chart
plt.plot([1,2,3],[1,4,9]); plt.title('implicit'); plt.close()

# The object-oriented API — always does what it says
fig, ax = plt.subplots(figsize=(6,3))
ax.plot([1,2,3],[1,4,9])
ax.set_title('explicit')
print('fig:', type(fig).__name__, '| ax:', type(ax).__name__)
plt.close(fig)


## 2. Distribution — try several bin counts


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13,3.5))
for ax, bins in zip(axes, [5, 40, 300]):
    ax.hist(df.revenue, bins=bins)
    ax.set_title(f'{bins} bins')
    ax.set_xlabel('Revenue (£)')
axes[0].set_ylabel('Customers')
fig.tight_layout()
print('Too few hides structure. Too many is noise.')
plt.close(fig)


Skewed data often only reveals its shape on a log scale.


In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11,3.5))
a1.hist(df.revenue, bins=50); a1.set_title('linear scale')
a2.hist(df.revenue, bins=50); a2.set_xscale('log'); a2.set_title('log scale')
fig.tight_layout(); plt.close(fig)
print(f'skew: {df.revenue.skew():.2f}  (0 is symmetric)')


## 3. Scatter — alpha is essential


In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11,4))
a1.scatter(df.age, df.revenue); a1.set_title('no alpha — a solid blob')
a2.scatter(df.age, df.revenue, alpha=0.2, s=8); a2.set_title('alpha=0.2 — density visible')
for a in (a1,a2): a.set_xlabel('Age'); a.set_ylabel('Revenue (£)')
fig.tight_layout(); plt.close(fig)


## 4. Time series — raw plus a rolling mean


In [ ]:
fig, ax = plt.subplots(figsize=(10,3.5))
ax.plot(daily.index, daily.values, alpha=0.35, lw=1, label='daily')
ax.plot(daily.index, daily.rolling(7).mean(), lw=2, label='7-day mean')
ax.set_ylabel('Revenue (£k)')
ax.set_title('Revenue grew steadily through H1 2026')   # the FINDING, not the subject
ax.legend(frameon=False)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout(); plt.close(fig)


## 5. Bars — horizontal, sorted, starting at zero


In [ ]:
counts = df.region.value_counts()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11,3.5))

a1.bar(sorted(counts.index), [counts[c] for c in sorted(counts.index)])
a1.set_title('alphabetical, vertical — harder to read')

a2.barh(counts.index[::-1], counts.values[::-1])
a2.set_title('sorted, horizontal — comparison is easy')
fig.tight_layout(); plt.close(fig)
print(counts)


## 6. Small multiples with a shared axis


In [ ]:
regions = sorted(df.region.unique())
fig, axes = plt.subplots(1, 4, figsize=(13,3), sharey=True)
for ax, r in zip(axes, regions):
    ax.hist(df.loc[df.region==r,'revenue'], bins=30)
    ax.set_title(r, fontsize=10)
axes[0].set_ylabel('Customers')
fig.tight_layout(); plt.close(fig)
print('sharey=True is what makes the panels comparable.')


## 7. Formatting numbers as a human would write them


In [ ]:
from matplotlib.ticker import FuncFormatter
fig, ax = plt.subplots(figsize=(7,3))
ax.plot(daily.index, daily.values * 1000)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'£{v:,.0f}'))
ax.set_title('Revenue')
fig.autofmt_xdate()
fig.tight_layout(); plt.close(fig)
print('Compare £1,234,567 against 1234567.0')


## 8. Exploratory charts should be fast and ugly


In [ ]:
axes = df.select_dtypes('number').hist(figsize=(9,3), bins=30, layout=(1,2))
plt.close('all')
print('One line, every numeric column. Disposable by design.')


## Try it yourself

1. Rewrite one title above as a finding rather than a label.
2. Build the same small multiples with `sharey=False` and explain why it misleads.
3. Save a figure with and without `bbox_inches='tight'` and compare the cropping.
